# 03 · Update, measure, gate, promote

*Part 2, sections 3-4.* The first full pass of the loop, by hand: a person writes v2, both versions
run on the same dataset with the same scorers, and v2 ships only if it measurably helps.

The rules every later level reuses (`selfimprove/gate.py`):

1. **must-pass** scorers are at 100% for the candidate, on both splits
2. the candidate is **not worse** than production on the train split (the CI suite)
3. it **beats production on the holdout** by at least `SI_PROMOTION_MARGIN`
4. it is better somewhere (a no-op never ships), and not bloated

In [ ]:
import _nb as nb

experiment_id = nb.setup()

In [ ]:
import mlflow
from selfimprove import agents, config, gate, instructions, signals, store

print(open("examples/fixes/v2_manual.md").read())

## 1. Register v2 without promoting it

The commit message names the evidence. `production` stays on v1 until v2 has been measured.

In [ ]:
v2 = mlflow.genai.register_prompt(
    name=config.settings.prompt_name,
    template=open("examples/fixes/v2_manual.md").read(),
    commit_message="v2: lead with the answer, always look up the policy, never guess "
                   "(evidence: the stale-refund and disclaimer failures in notebook 01)",
    tags={"author": "you", "status": "candidate"},
)
print(instructions.diff("production", v2.version))

## 2. Measure both on the same data

Each `evaluate()` call is an MLflow evaluation run, so the two sit side by side in the UI with
per-row outcomes and rationales. `predict_fn` pins a version: the agent answers every row fresh.

In [ ]:
train, holdout = store.rows("train"), store.rows("holdout")
scorers = signals.full()
print(len(train), "train rows,", len(holdout), "holdout rows;", [s.name for s in scorers])

v1_train = mlflow.genai.evaluate(data=train, predict_fn=agents.predict_fn(1), scorers=scorers)
v2_train = mlflow.genai.evaluate(data=train, predict_fn=agents.predict_fn(v2.version), scorers=scorers)

In [ ]:
import pandas as pd

def pass_rates(result):
    df = result.result_df
    return {s.name: df[f"{s.name}/value"].map(gate.passed).mean() for s in scorers if f"{s.name}/value" in df}

pd.DataFrame({"v1": pass_rates(v1_train), "v2": pass_rates(v2_train)}).style.format("{:.0%}")

Three things to check before trusting the numbers: the cases that failed should now pass (the
change targeted them), the golden cases should still pass, and the **raw responses** should
actually be good, not just pass:

In [ ]:
for _, row in v2_train.result_df.head(4).iterrows():
    print("Q:", row["request"]["question"])
    print("A:", str(row["response"])[:240], "\n")

## 3. The gate

`gate.compare()` does the above for both splits and applies the rules. It is the same function
that gates the analyst's, the optimizer's and a coding agent's candidates later.

In [ ]:
decision = gate.compare(v2.version, "production")
print(decision.table())
print()
print(decision.summary())

## 4. The dataset as a regression suite

The same rows run as a pytest suite with `@mlflow.test`, one MLflow run per session, and CI runs
it on every pull request (`.github/workflows/selfimprove-regression.yml`):

In [ ]:
print(open("tests/regression/test_regressions.py").read().split('"""', 2)[2])

```bash
SI_REGRESSION=1 pytest -p mlflow.pytest.plugin tests/regression -n 4
```

## 5. Promote: a pointer move

The agent picks up v2 on its next request, with no code change. The version it replaced becomes
`previous`, so rollback is one alias move too.

In [ ]:
if decision.passed:
    replaced = instructions.promote(v2.version, reason=f"gate run {decision.run_id}", by="you")
    print(f"production: v{replaced} -> v{instructions.load('production').version}")
else:
    print("not promoting:", decision.reasons)

In [ ]:
print(agents.run("How long can I wait to request a refund?", source="production"))

One pass produced two prompt versions, a dataset, two evaluation runs that show the before and
after, and a regression suite. Every step was manual.

**Next:** [04 · Triage and approve](04_triage_and_approve.ipynb), for when there is too much signal
to read by hand.